# Module 5 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

## Solving Normal Form Games

In [1]:
from typing import List, Tuple, Dict, Callable
import copy

In the lecture we talked about the Prisoner's Dilemma game, shown here in Normal Form:

Player 1 / Player 2  | Defect | Cooperate
------------- | ------------- | -------------
Defect  | -5, -5 | -1, -10
Cooperate  | -10, -1 | -2, -2

where the payoff to Player 1 is the left number and the payoff to Player 2 is the right number. We can represent each payoff cell as a Tuple: `(-5, -5)`, for example. We can represent each row as a List of Tuples: `[(-5, -5), (-1, -10)]` would be the first row and the entire table as a List of Lists:

In [2]:
prisoners_dilemma = [
 [( -5, -5), (-1,-10)],
 [(-10, -1), (-2, -2)]]

prisoners_dilemma

[[(-5, -5), (-1, -10)], [(-10, -1), (-2, -2)]]

in which case the strategies are represented by indices into the List of Lists. For example, `(Defect, Cooperate)` for the above game becomes `prisoners_dilemma[ 0][ 1]` and returns the payoff `(-1, -10)` because 0 is the first row of the table ("Defect" for Player 1) and 1 is the 2nd column of the row ("Cooperate" for Player 2).

For this assignment, you are going write a function that uses Successive Elimination of Dominated Strategies (SEDS) to find the **pure strategy** Nash Equilibrium of a Normal Form Game. The function is called `solve_game`:

```python
def solve_game( game: List[List[Tuple]], weak=False) -> List[Tuple]:
    pass # returns strategy indices of Nash equilibrium or None.
```

and it takes two parameters: the game, in a format that we described earlier and an optional boolean flag that controls whether the algorithm considers only **strongly dominated strategies** (the default will be false) or whether it should consider **weakly dominated strategies** as well.

It should work with game matrices of any size and it will return the **strategy indices** of the Nash Equilibrium. If there is no **pure strategy** equilibrium that can be found using SEDS, return the empty List (`[]`).


<div style="background: mistyrose; color: firebrick; border: 2px solid darkred; padding: 5px; margin: 10px;">
Do not return the payoff. That's not useful. Return the strategy indices, any other output is incorrect.
</div>

As before, you must provide your implementation in the space below, one Markdown cell for documentation and one Code cell for implementation, one function and assertations per Codecell.


---

<a id="is_dominated"></a>
## is_dominated

To find a Nash equilibrium, dominated strategies are found and removed. A strategy can be weakly dominated or strongly dominated. A strongly dominated strategy is when another strategy has a better outcome for every possible strategy of the other player. A weakly dominated strategy is when another strategy has an equal or better outcome for every possible strategy of the other player. 

This function checks whether a specific row, or column in the normal form game is dominated by the other rows, or columns. If weak = True, weakly dominated strategies are also considered, in addition to strongly dominated strategies.

* **strategy_index** int: index of the row or column to be checked if dominated
* **player** int: 0 or 1, the player whose strategy is being evaluated. This will determine whether a row or column of the normal form game is being evaluated.
* **game** List[List[Tuple]]: the grid of payoff for each strategy played by each player
* **weak** boolean: True if checking for weakly dominated strategies in addition to strongly dominated strategies.
  
**returns** **is_dominated** boolean: true if the row/column in question is dominated by another row/column, false if not

In [3]:
def is_dominated(strategy_index: int, player: int, game: List[List[Tuple]], weak: bool):
    dominated = False
    strategy_count = [len(game), len(game[0])]  #number of strategies left for each player
    for comparison_index in range(0, strategy_count[player]):
        if comparison_index == strategy_index:
            continue

        #player 0/1 determines whether to iterate row or column in game
        #weak true/false determines the dominant strategy used
        if player == 0:        
            if weak: 
                dominated = all(game[comparison_index][k][0] >= game[strategy_index][k][0] for k in range(strategy_count[1]))
            else:
                dominated = all(game[comparison_index][k][0] > game[strategy_index][k][0] for k in range(strategy_count[1]))
        else: 
            if weak: 
                dominated = all(game[k][comparison_index][1] >= game[k][strategy_index][1] for k in range(strategy_count[0]))
            else: 
                dominated = all(game[k][comparison_index][1] > game[k][strategy_index][1] for k in range(strategy_count[0]))

        if dominated: 
            return True
    
    return False

In [4]:
#unit test
prisoners_dilemma = [
 [( -5, -5), (-1,-10)],
 [(-10, -1), (-2, -2)]]

assert is_dominated(0, 0, prisoners_dilemma, True) == False
assert is_dominated(1, 0, prisoners_dilemma, True) == True
assert is_dominated(1, 1, prisoners_dilemma, True) == True
assert is_dominated(0, 1, prisoners_dilemma, True) == False

<a id="eliminate_dominated"></a>
## eliminate_dominated

In the search for a Nash equilibrium, each player's strategies are evaluated to find an optimal strategy, given the other player's chosen strategy. If the player does not have any incentive to change their strategy, it is the optimal strategy. In a Nash equilbrium, neither of the players can acheive a better outcome if they change their strategy. The search for the Nash equilibrium is done by eliminating strategies that are dominated by other strategies. A strategy is dominated when another strategy has a better outcome. 

This function recursively evaluates the payoff grid to remove dominated strategies until an equilibrium is found, if one is available. This function designed to find all Nash equilbriums for a given payoff grid. The function is a recursive depth-first method to evaluate all possible options. 

* **game** List[List[Tuple]]: the grid of payoff for each strategy played by each player
* **weak** boolean: True if checking for weakly dominated strategies in addition to strongly dominated strategies.
* **player** int: 0 or 1, the player whose strategy is being evaluated. This will determine whether a row or column of the normal form game is being evaluated.
* **equilibrium_strategies** List[Tuple]: list of Nash equilibrium strategies found so far
* **player_0_strategies** List[int]: list of strategies remaining for first player
* **player_1_strategies** List[int]: list of strategies remaining for second player

**returns** **equilibrium_strategies** List[Tuple]: list of Nash equilibrium strategies found so far

In [5]:
def eliminate_dominated(game: List[List[Tuple]], weak: bool, player: int, equilibirum_strategies: List[Tuple], player_0_strategies: List[int], player_1_strategies: List[int]):
    strategy_count = [len(game), len(game[0])]  #number of strategies left for eahc player

    #equilibrium found when only one strategy left for each player after elimination
    if strategy_count[0] == 1 and strategy_count[1] == 1:
        equilibirum_strategies.extend([(i, j) for i in player_0_strategies for j in player_1_strategies])
        return equilibirum_strategies

    #identify all dominated strategies for the player
    is_dominated_list = []
    for strategy_index in range(strategy_count[player]):
        dominated = is_dominated(strategy_index, player, game, weak)
        if dominated: 
            is_dominated_list.append(strategy_index)

    #if no dominated strategies, exit 
    if len(is_dominated_list) == 0: 
         return equilibirum_strategies

    # remove dominated strategies and recursively use eliminate_dominated to continue removing strategies 
    # depth-first approach used - is_dominated_list acts as a frontier
    else:
        for strategy_index in is_dominated_list:
                game_copy = copy.deepcopy(game)
                player_0_copy = copy.deepcopy(player_0_strategies)
                player_1_copy = copy.deepcopy(player_1_strategies)
            
                if player == 0:
                    del game_copy[strategy_index]
                    del player_0_copy[strategy_index]
                else:
                    for row in game_copy:
                        del row[strategy_index]
                    del player_1_copy[strategy_index]
                    
                equilibrium_new = eliminate_dominated(game_copy, weak, 1 - player, equilibirum_strategies, player_0_copy, player_1_copy)
                equilibirum_strategies.extend(equilibrium_new)

    return equilibirum_strategies

In [6]:
#unit test
prisoners_dilemma = [
 [( -5, -5), (-1,-10)],
 [(-10, -1), (-2, -2)]]

test_player_0_strategies = list(range(len(prisoners_dilemma)))
test_player_1_strategies = list(range(len(prisoners_dilemma[0])))
test_equilibirum_strategies = []

strategies = eliminate_dominated(prisoners_dilemma, True, 0, test_equilibirum_strategies, test_player_0_strategies, test_player_1_strategies)
strategies = list(set(strategies))
assert strategies == [(0,0)]
assert len(prisoners_dilemma) == len(prisoners_dilemma[0])
assert len(strategies) == 1

<a id="solve_game"></a>
## solve_game

In the search for a Nash equilibrium, each player's strategies are evaluated to find an optimal strategy, given the other player's chosen strategy. If the player does not have any incentive to change their strategy, it is the optimal strategy. In a Nash equilbrium, neither of the players can acheive a better outcome if they change their strategy. The search for the Nash equilibrium is done by eliminating strategies that are dominated by other strategies. A strategy is dominated when another strategy has a better outcome. 

This function initiate the search for all Nash equilibrium. It evaluates both scenarios of when Player 0 starts or Player 1 starts. 

* **game** List[List[Tuple]]: the grid of payoff for each strategy played by each player
* **weak** boolean: True if checking for weakly dominated strategies in addition to strongly dominated strategies

**returns** **equilibrium_strategies** List[Tuple]: list of Nash equilibrium strategies found in the game

In [7]:
def solve_game(game: List[List[Tuple[int, int]]], weak: bool = False) -> List[Tuple[int, int]]:
    nash_equilibrium = list()
    
    # used to preserve the strategies indices as they are deleted
    player_0_strategies = list(range(len(game)))
    player_1_strategies = list(range(len(game[0])))
    
    for player in range(0,2):  #attempt starting with player 1 or player 2
        equilibirum_strategies = []
        equilibrium = eliminate_dominated(game, weak, player, equilibirum_strategies, player_0_strategies, player_1_strategies)
        nash_equilibrium.extend(equilibrium)

    nash_equilibrium = list(set(nash_equilibrium))  #find unique strategies

    return nash_equilibrium  #returns a list of tuples

In [8]:
#unit test
prisoners_dilemma = [
 [( -5, -5), (-1,-10)],
 [(-10, -1), (-2, -2)]]


assert len(solve_game(prisoners_dilemma, True)[0])
assert solve_game(prisoners_dilemma, True) == [(0, 0)]
assert solve_game(prisoners_dilemma, False) == [(0, 0)]

## Additional Directions

Create three games as described and according to the following:

1. Your games must be created and solved "by hand".
2. The strategy pairs must **not** be on the main diagonal (0, 0), (1, 1), or (2, 2). And the solution cannot be the same for both Game 1 and Game 2.
3. Make sure you fill out the Markdown ("?") with your game as well as the solution ("?").
4. Remember, **do not return the payoff**, return the strategy indices (a list of them).

## Before you code...

Solve the following game by hand using SEDS and weakly dominated strategies. 
The game has three (pure) Nash Equilibriums. 
You should find all of them.
This will help you think about what you need to implement to make the algorithm work.
**Hint**: You will need State Space Search from Module 1 and SEDS from Module 5 to get the full algorithm to work.

| Player 1 / Player 2  | 0 | 1 | 2 |
|----|----|----|----|
|0  | 1/0 | 3/1 | 1/1 |
|1  | 1/1 | 3/0 | 0/1 |
|2  | 2/2 | 3/3 | 0/2 |

**Solutions**:

##### Solution 1

Step 1: Evaluate Player 1

Strategy 1 is weakly dominated by Strategy 2. Strategy 1 can be removed. 
| Player 1 / Player 2  | 0 | 1 | 2 |
|----|----|----|----|
|0  | 1/0 | 3/1 | 1/1 |
|2  | 2/2 | 3/3 | 0/2 |

Step 2: Evaluate Player 2

Strategy 0 is weakly dominated by Strategy 1. Strategy 0 can be removed.

| Player 1 / Player 2  | 1 | 2 |
|----|----|----|
|0  | 3/1 | 1/1 |
|2  | 3/3 | 0/2 |

Step 3: Evaluate Player 1

Strategy 0 is strongly dominated by Strategy 2. Strategy 0 can be removed.

| Player 1 / Player 2  | 1 | 2 |
|----|----|----|
|2  | 3/3 | 0/2 |

Step 4: Evaluate Player 2

Strategy 2 is strongly dominated by Strategy 1. Strategy 2 can be removed.

| Player 1 / Player 2  | 1 |
|----|----|
|2  | 3/3 |

Equilibrium: (2, 1) - where player 1 plays strategy 2, and player 2, plays strategy 1


##### Solution 2

Step 1: Evaluate Player 2

Strategy 1 is strongly dominated by Strategy 2. Strategy 1 can be removed. 

| Player 1 / Player 2  | 0 | 2 |
|----|----|----|
|0  | 1/0 | 1/1 |
|1  | 1/1 | 0/1 |
|2  | 2/2 | 0/2 |

Step 2: Evaluate Player 1

Strategy 1 is weakly dominated by Strategy 2. Strategy 1 can be removed.

| Player 1 / Player 2  | 0 | 2 |
|----|----|----|
|0  | 1/0 | 1/1 |
|2  | 2/2 | 0/2 |

Step 3: Evaluate Player 2

Strategy 0 is weakly dominated by Strategy 2. Strategy 0 can be removed.

| Player 1 / Player 2  | 2 |
|----|----|
|0  | 1/1 |
|2  | 0/2 |

Step 4: Evaluate Player 1

Strategy 2 is strongly dominated by Strategy 0. Strategy 2 can be removed.

| Player 1 / Player 2  | 2 |
|----|----|
|0  | 1/1 |

Equilibrium: (0, 2) - where player 1 plays strategy 0, and player 2, plays strategy 2


##### Solution 3

Step 1: Evaluate Player 2

Strategy 0 is weakly dominated by Strategy 2. Strategy 0 can be removed. 

| Player 1 / Player 2  | 1 | 2 |
|----|----|----|
|0  | 3/1 | 1/1 |
|1  | 3/0 | 0/1 |
|2  | 3/3 | 0/2 |

Step 2: Evaluate Player 1

Strategy 1 is strongly dominated by Strategy 2. Strategy 1 can be removed.
| Player 1 / Player 2  | 1 | 2 |
|----|----|----|
|0  | 3/1 | 1/1 |
|2  | 3/3 | 0/2 |

Step 3: Evaluate Player 1

Strategy 2 is weakly dominated by Strategy 0. Strategy 2 can be removed.

| Player 1 / Player 2  | 1 | 2 |
|----|----|----|
|0  | 3/1 | 1/1 |

Step 4: Evaluate Player 2

Strategy 2 is strongly dominated by Strategy 1. Strategy 2 can be removed.

| Player 1 / Player 2  | 1 |
|----|----|
|0  | 3/1 |

Equilibrium: (0, 1) - where player 1 plays strategy 0, and player 2, plays strategy 1

#### All Solutions: (0, 1), (0, 2), (2, 1)


### Test Game 1. Create a 3x3 two player game

**that can only be solved using the Successive Elimintation of Strongly Dominated Strategies**

| Player 1 / Player 2  | 0 | 1 | 2 |
|----|----|----|----|
|0  | 3/0 | 4/1 | 9/9 |
|1  | 2/1 | 2/0 | 0/3 |
|2  | 2/2 | 3/3 | 1/2 |

**Solution:**

Step 1: Evaluate Player 1

Strategy 1 is strongly dominated by Strategy 0. Strategy 1 can be removed. 

| Player 1 / Player 2  | 0 | 1 | 2 |
|----|----|----|----|
|0  | 3/0 | 4/1 | 9/9 |
|2  | 2/2 | 3/3 | 1/2 |

Step 2: Evaluate Player 2

Strategy 0 is strongly dominated by Strategy 1. Strategy 0 can be removed.
| Player 1 / Player 2  | 1 | 2 |
|----|----|----|
|0  | 4/1 | 9/9 |
|2  | 3/3 | 1/2 |

Step 3: Evaluate Player 1

Strategy 2 is strongly dominated by Strategy 0. Strategy 2 can be removed.

| Player 1 / Player 2  | 1 | 2 |
|----|----|----|
|0  | 4/1 | 9/9 |

Step 4: Evaluate Player 2

Strategy 1 is strongly dominated by Strategy 2. Strategy 1 can be removed.

| Player 1 / Player 2  | 2 |
|----|----|
|0  | 9/9 |

Equilibrium: (0, 2) - where player 1 plays strategy 0, and player 2, plays strategy 2

In [9]:
test_game_1 = [
[(3, 0), (4, 1), (9, 9)],
[(2, 1), (2, 0), (0, 3)],
[(2, 2), (3, 3), (1, 2)]]

solution = solve_game(test_game_1)

In [10]:
assert solution == [(0,2)] # insert your solution from above.

### Test Game 2. Create a 3x3 two player game

**that can only be solved using the Successive Elimintation of Weakly Dominated Strategies**

| Player 1 / Player 2  | 0 | 1 | 2 |
|----|----|----|----|
|0  | 3/2 | 4/2 | 4/2 |
|1  | 3/2 | 4/2 | 1/1 |
|2  | 2/2 | 5/5 | 2/3 |

**Solution:**

Step 1: Evaluate Player 1

Strategy 1 is weakly dominated by Strategy 0. Strategy 1 can be removed. 

| Player 1 / Player 2  | 0 | 1 | 2 |
|----|----|----|----|
|0  | 3/2 | 4/2 | 4/2 |
|2  | 2/2 | 5/5 | 2/3 |

Step 2: Evaluate Player 2

Strategy 2 is weakly dominated by Strategy 0. Strategy 2 can be removed.

| Player 1 / Player 2  | 0 | 1 |
|----|----|----|
|0  | 3/2 | 4/2 |
|2  | 2/2 | 5/5 |

Step 3: Evaluate Player 2

Strategy 0 is weakly dominated by Strategy 1. Strategy 0 can be removed.

| Player 1 / Player 2  | 1 |
|----|----|
|0  | 4/2 |
|2  | 5/5 |

Step 4: Evaluate Player 1

Strategy 0 is strongly dominated by Strategy 2. Strategy 0 can be removed.

| Player 1 / Player 2  | 1 |
|----|----|
|2  | 5/5 |

Equilibrium: (2, 1) - where player 1 plays strategy 0, and player 2, plays strategy 2

In [11]:
test_game_2 = [
[(3, 2), (4, 2), (4, 2)],
[(3, 2), (4, 2), (1, 1)],
[(2, 2), (5, 5), (2, 3)]]

strong_solution = solve_game(test_game_2)
weak_solution = solve_game(test_game_2, weak=True)

In [12]:
assert strong_solution == []
assert weak_solution == [(2, 1)] # insert your solution from above.

### Test Game 3. Create a 3x3 two player game

**that cannot be solved using the Successive Elimintation of Dominated Strategies at all**

| Player 1 / Player 2  | 0 | 1 | 2 |
|----|----|----|----|
|0  | 1/4 | 2/5 | 3/6 |
|1  | 3/6 | 2/5 | 1/4 |
|2  | 2/5 | 5/5 | 2/5 |

**Solution:** None

In [13]:
test_game_3 = [
[(1, 4), (2, 5), (3, 6)],
[(3, 6), (2, 5), (1, 4)],
[(2, 5), (5, 5), (2, 5)]]

strong_solution = solve_game(test_game_3)
weak_solution = solve_game(test_game_3, weak=True)

In [14]:
assert strong_solution == []
assert weak_solution == []

### Test Game 4. Multiple Equilibria

You solve the following game by hand, above.
Now use your code to solve it.

| Player 1 / Player 2  | 0 | 1 | 2 |
|----|----|----|----|
|0  | 1/0 | 3/1 | 1/1 |
|1  | 1/1 | 3/0 | 0/1 |
|2  | 2/2 | 3/3 | 0/2 |

**Solutions:** [(0, 1), (0, 2), (2, 1)]

In [15]:
test_game_4 = [
[(1, 0), (3, 1), (1, 1)],
[(1, 1), (3, 0), (0, 3)],
[(2, 2), (3, 3), (0, 2)]]

strong_solution = solve_game(test_game_4)
weak_solution = solve_game(test_game_4, weak=True)

In [16]:
assert strong_solution == []
assert weak_solution == [(0, 1), (0, 2), (2, 1)] # put solution here

## Before You Submit...

1. Did you provide output exactly as requested? **Don't forget to fill out the Markdown tables with your games**.
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.